In [1]:
import matplotlib.pyplot as plt
import h5py
import numpy as np
import glob

In [2]:
def readfile(filename):
    
    
    with h5py.File(filename, 'r') as ar:
        name = filename.split('/')[2]

        iteration_number = 15 #25 # total n iterations
        tot = iteration_number*6
        Sigma_iw=ar['dmft_output']['Sigma_iw']
        freq=np.linspace(-161,161,2050)
        mask = (freq > 0) & (freq < 10)
        freq_sel = freq[mask]


        nwarmupstep = int(name.split('.')[0].split('_')[2][7:])
        list_nwarmup = np.repeat(nwarmupstep,tot)
        real_all_sigma = np.zeros([tot,len(freq_sel)])
        img_all_sigma = np.zeros([tot,len(freq_sel)])

        real_all_g0 = np.zeros([tot,len(freq_sel)])
        img_all_g0 = np.zeros([tot,len(freq_sel)])
    
    
        count =0
        spin_list =['down_0','down_1','down_2','up_0','up_1','up_2']
        for it in range(1,iteration_number+1):
            
            chunk = ar['dmft_output']['Sigma-%s'%(it)]
            chunk2 = ar['dmft_output']['G0-%s'%(it)]
            for spin in spin_list:
                freq = Sigma_iw[spin]['mesh']['domain']
                data1 = np.asarray(chunk[spin]['data'][()]).reshape(2050,2)
                data_real_sigma,data_img_sigma = data1[:,0], data1[:,1]
                data_real_sigma_sel = data_real_sigma[mask]
        
                data_img_sigma_sel = data_img_sigma[mask]
                real_all_sigma[count] = data_real_sigma_sel
                img_all_sigma[count] = data_img_sigma_sel


                data2 = np.asarray(chunk2[spin]['data'][()]).reshape(2050,2)
                data_real_g0,data_img_g0 = data2[:,0], data2[:,1]
                data_real_g0_sel = data_real_g0[mask]
                data_img_g0_sel = data_img_g0[mask]
                real_all_g0[count] = data_real_g0_sel
                img_all_g0[count] = data_img_g0_sel



                
                count = count+1
        return list_nwarmup, real_all_sigma,img_all_sigma,real_all_g0,img_all_g0
  

In [7]:
#### for lvo, read the last iteration to get rid of noisy files

def readfile_special(filename):
    
    
    with h5py.File(filename, 'r') as ar:
        name = filename.split('/')[2]

        iteration_number = 3 #25 # total n iterations
        tot = 6
        Sigma_iw=ar['dmft_output']['Sigma_iw']
        freq=np.linspace(-161,161,2050)
        mask = (freq > 0) & (freq < 10)
        freq_sel = freq[mask]


        nwarmupstep = int(name.split('.')[0].split('_')[2][7:])
        list_nwarmup = np.repeat(nwarmupstep,tot)
        real_all_sigma = np.zeros([tot,len(freq_sel)])
        img_all_sigma = np.zeros([tot,len(freq_sel)])

        real_all_g0 = np.zeros([tot,len(freq_sel)])
        img_all_g0 = np.zeros([tot,len(freq_sel)])
    
    
        count =0
        spin_list =['down_0','down_1','down_2','up_0','up_1','up_2']
        
        chunk = ar['dmft_output']['Sigma-%s'%(iteration_number)]
        chunk2 = ar['dmft_output']['G0-%s'%(iteration_number)]
        for spin in spin_list:
            freq = Sigma_iw[spin]['mesh']['domain']
            data1 = np.asarray(chunk[spin]['data'][()]).reshape(2050,2)
            data_real_sigma,data_img_sigma = data1[:,0], data1[:,1]
            data_real_sigma_sel = data_real_sigma[mask]
            data_img_sigma_sel = data_img_sigma[mask]
            real_all_sigma[count] = data_real_sigma_sel
            img_all_sigma[count] = data_img_sigma_sel


            data2 = np.asarray(chunk2[spin]['data'][()]).reshape(2050,2)
            data_real_g0,data_img_g0 = data2[:,0], data2[:,1]
            data_real_g0_sel = data_real_g0[mask]
            data_img_g0_sel = data_img_g0[mask]
            real_all_g0[count] = data_real_g0_sel
            img_all_g0[count] = data_img_g0_sel



                
            count = count+1
        return list_nwarmup, real_all_sigma,img_all_sigma,real_all_g0,img_all_g0
  

In [3]:


allfiles = glob.glob('./sims_CVO/*')
print(len(allfiles))
nwarmup_lists =[]
real_sigma_all =[]
img_sigma_all =[]

real_g0_all =[]
img_g0_all =[]
for mf in allfiles:
    try:
        nwarmup, real_sigma, img_sigma, real_g0,img_g0 = readfile(mf)
        
        for x,y,z,m,n in zip(real_sigma,img_sigma,nwarmup,real_g0,img_g0):
            real_sigma_all.append(x)
            img_sigma_all.append(y)
            nwarmup_lists.append(z)

            real_g0_all.append(m)
            img_g0_all.append(n)
        
        
    except:
        print('something wrong for file', mf)



6


In [4]:
nwarmup_lists = np.array(nwarmup_lists)
real_all_sigma = np.asarray(real_sigma_all)
img_all_sigma = np.asarray(img_sigma_all)
real_all_g0 = np.asarray(real_g0_all)
img_all_g0 = np.asarray(img_g0_all)
indices = np.arange(real_all_sigma.shape[0])
# Shuffle the indices
np.random.shuffle(indices)

# Use the shuffled indices to shuffle both arrays in the same order
real_shuff_sigma = real_all_sigma[indices]
img_shuff_sigma = img_all_sigma[indices]

real_shuff_g0 = real_all_g0[indices]
img_shuff_g0 = img_all_g0[indices]
nwarmup_shuff = nwarmup_lists[indices]
#print(nwarmup_shuff)



In [11]:
np.savez_compressed('dataCVO.npz',nwarupsteps=nwarmup_shuff, realpart_sigma=real_shuff_sigma,imgpart_sigma=img_shuff_sigma,realpart_g0=real_shuff_g0,imgpart_g0=img_shuff_g0)

In [ ]:
print (np.arange(1,5+1))